In [1]:

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"D:\RouteRanker")
HYDERABAD_DIR = PROJECT_DIR / "data" / "raw" / "Hyderabad"

# Locate the folder containing routes.txt
matches = list(HYDERABAD_DIR.rglob("routes.txt"))

if not matches:
    print("ERROR: routes.txt not found. Check the dataset folder.")
else:
    GTFS_DIR = matches[0].parent
    print("Hyderabad GTFS folder:", GTFS_DIR)

    routes = pd.read_csv(GTFS_DIR / "routes.txt", dtype=str)
    stops = pd.read_csv(GTFS_DIR / "stops.txt", dtype=str)
    trips = pd.read_csv(GTFS_DIR / "trips.txt", dtype=str)

    print("\nRoutes shape:", routes.shape)
    print("Stops shape:", stops.shape)
    print("Trips shape:", trips.shape)

    stop_times_path = GTFS_DIR / "stop_times.txt"

    if stop_times_path.exists():
        stop_times = pd.read_csv(stop_times_path, dtype=str)
        print("Stop times shape:", stop_times.shape)
    else:
        print("WARNING: stop_times.txt not found")


Hyderabad GTFS folder: D:\RouteRanker\data\raw\Hyderabad\telangana_opendata_gtfs_tgsrtc_08_february_2026

Routes shape: (1031, 4)
Stops shape: (5028, 6)
Trips shape: (44892, 5)
Stop times shape: (1132807, 6)


In [2]:

print("===== HYDERABAD GTFS RELATIONSHIP CHECKS =====")

print(
    "Trips with missing routes:",
    (~trips["route_id"].isin(routes["route_id"])).sum()
)

print(
    "Stop times with missing trips:",
    (~stop_times["trip_id"].isin(trips["trip_id"])).sum()
)

print(
    "Stop times with missing stops:",
    (~stop_times["stop_id"].isin(stops["stop_id"])).sum()
)

print(
    "Trips without stop-time records:",
    (~trips["trip_id"].isin(stop_times["trip_id"])).sum()
)

print(
    "Duplicate stop-time rows:",
    stop_times.duplicated().sum()
)


===== HYDERABAD GTFS RELATIONSHIP CHECKS =====
Trips with missing routes: 0
Stop times with missing trips: 0
Stop times with missing stops: 0
Trips without stop-time records: 0
Duplicate stop-time rows: 0


In [3]:

print("===== HYDERABAD DATA QUALITY =====")

print("\nMissing values in routes:")
display(routes.isnull().sum())

print("\nMissing values in stops:")
display(stops.isnull().sum())

print("\nMissing values in trips:")
display(trips.isnull().sum())

print("\nMissing values in stop_times:")
display(stop_times.isnull().sum())


===== HYDERABAD DATA QUALITY =====

Missing values in routes:


route_id           0
route_long_name    0
agency_id          0
route_type         0
dtype: int64


Missing values in stops:


stop_id      0
stop_name    0
zone_id      0
stop_lat     0
stop_lon     0
stop_desc    0
dtype: int64


Missing values in trips:


route_id           0
trip_id            0
service_id         0
direction_id       0
trip_short_name    0
dtype: int64


Missing values in stop_times:


trip_id           0
stop_sequence     0
stop_id           0
departure_time    0
arrival_time      0
timepoint         0
dtype: int64

In [2]:

from pathlib import Path
import pandas as pd

BASE = Path(r"D:\RouteRanker\data\raw\chennai")
matches = list(BASE.rglob("routes.txt"))

if matches:
    GTFS_DIR = matches[0].parent

    for path in sorted(GTFS_DIR.glob("*.txt")):
        df = pd.read_csv(path, dtype=str)
        print(f"\n{path.name}: {len(df)} rows, {len(df.columns)} columns")
        print("Columns:", list(df.columns))
        print("Missing values:", int(df.isna().sum().sum()))
else:
    print("routes.txt not found. Check the Chennai folder.")



agency.txt: 1 rows, 6 columns
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_phone', 'agency_lang']
Missing values: 0

calendar.txt: 6 rows, 10 columns
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
Missing values: 0

feed_info.txt: 1 rows, 8 columns
Columns: ['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'feed_start_date', 'feed_end_date', 'feed_version', 'feed_contact_email', 'feed_contact_url']
Missing values: 2

routes.txt: 4611 rows, 5 columns
Columns: ['route_id', 'route_short_name', 'route_long_name', 'route_type', 'agency_id']
Missing values: 0

stop_times.txt: 1360443 rows, 5 columns
Columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence']
Missing values: 0

stops.txt: 5580 rows, 4 columns
Columns: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon']
Missing values: 0

trips.txt: 47047 rows, 4 columns
Columns: ['trip_id', 'route_id


## Dataset Relationships

This dataset follows the General Transit Feed Specification (GTFS), which organizes public transit information into related files.

- **routes.txt:** Contains route information. Routes are identified by `route_id`.
- **trips.txt:** Contains scheduled trips. The `route_id` connects a trip to a route, while `service_id` identifies its service schedule.
- **stop_times.txt:** Contains scheduled stop times and stop order for each trip. The `trip_id` connects records to trips, and `stop_id` connects them to stops.
- **stops.txt:** Contains stop identifiers, names, and available location information.
- **calendar.txt:** When present, defines regular service days and operating periods.
- **calendar_dates.txt:** When present, specifies additions or exceptions to regular service dates.

### Relationship Validation

The notebook checks whether trips reference valid routes, stop-time records reference valid trips and stops, and trips have stop-time records. Duplicate identifiers and missing values are also examined.
